# Week 3, day 2 — Extra practice 09 SOLUTIONS: outliers   (L05)

Executed in the lab image (pandas 3.0.5) against the real
`../data/orders_long.csv`. Every quoted number is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 09 — Outliers. Run this once.
import pandas as pd

orders = pd.read_csv("../data/orders_long.csv")

print(orders[["Sales", "Profit", "Quantity"]].describe().round(2).to_string())

### Question 1

`Q1 -54.61`, `Q3 226.73`, `IQR 281.34`, fences `-476.62` to `648.74`. -> **`31` below**, **`146` above**, `177 of 1093` — **16.2%**.

Unlike `Sales`, this column has a populated lower tail, because profit can
genuinely be negative. `Q1` itself is below zero.

16.2% flagged is even more than the 11.3% on `Sales`. Again the rule is
reporting the shape of the distribution rather than finding anomalies — and
a rule that flags one row in six is not describing rare events.

In [ ]:
p = orders["Profit"]
q1, q3 = p.quantile(0.25), p.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print("Q1 %.2f | Q3 %.2f | IQR %.2f" % (q1, q3, iqr))
print("fences: %.2f to %.2f" % (lower, upper))
print()
print("below:", int((p < lower).sum()), "| above:", int((p > upper).sum()))
print("total flagged: %d of %d (%.1f%%)"
      % (((p < lower) | (p > upper)).sum(), len(p),
         100 * ((p < lower) | (p > upper)).sum() / len(p)))

### Question 2

The extremes are ordinary-looking orders — large losses and large gains, with sales to match.

No impossible values, no obvious data entry errors. Big orders make big
profits and big losses, which is what you would expect.

The review step is cheap and it is the only thing that distinguishes 'this
value is extreme' from 'this value is wrong'.

In [ ]:
cols = ["OrderID", "Category", "Sales", "Profit"]
print("largest losses:")
print(orders.nsmallest(5, "Profit")[cols].to_string(index=False))
print()
print("largest gains:")
print(orders.nlargest(5, "Profit")[cols].to_string(index=False))

### Question 3

Per-category fences differ hugely: Office Supplies `1049.07`, Technology `6834.98`, Furniture `7894.12`. -> flagged `84`, `25`, `13`.

The Furniture fence is more than seven times the Office Supplies fence,
because furniture orders are simply larger.

A single global fence set from the mixture therefore means something
different in each category: it is generous to Office Supplies and harsh on
Furniture. Fencing within group asks the right question — 'unusual for its
kind' rather than 'large in absolute terms'.

In [ ]:
for cat, grp in orders.groupby("Category"):
    s = grp["Sales"]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    upper = q3 + 1.5 * (q3 - q1)
    print("%-16s Q1 %8.2f  Q3 %9.2f  fence %10.2f  flagged %3d of %4d"
          % (cat, q1, q3, upper, int((s > upper).sum()), len(s)))

### Question 4

Global flags `123`, per-category flags `122`. -> only **`56` flagged by both**; `67` only globally and `66` only per category.

Almost the same **number** of rows and largely **different** rows. Fewer
than half overlap.

That is the point worth making to anyone who thinks outlier detection is
objective. Two reasonable implementations of the same idea, producing counts
within one of each other, and disagreeing about which specific orders are
unusual more than half the time.

If the two totals had differed wildly someone would investigate. Because
they nearly match, nobody checks the membership.

In [ ]:
s = orders["Sales"]
q1, q3 = s.quantile(0.25), s.quantile(0.75)
global_mask = s > q3 + 1.5 * (q3 - q1)

per_cat = orders.groupby("Category")["Sales"].transform(
    lambda x: x > x.quantile(0.75) + 1.5 * (x.quantile(0.75) - x.quantile(0.25)))

print("flagged globally:     ", int(global_mask.sum()))
print("flagged per category: ", int(per_cat.sum()))
print("flagged by both:      ", int((global_mask & per_cat).sum()))
print("only by the global:   ", int((global_mask & ~per_cat).sum()))
print("only per category:    ", int((~global_mask & per_cat).sum()))

### Question 5

`66` orders flagged per category but not globally — mostly Office Supplies, well below the global fence.

These are large Office Supplies orders: unremarkable against the whole file,
but far out for their own category.

Whether they deserve flagging depends entirely on the question. For 'find
data entry errors', probably not. For 'find unusual purchasing behaviour',
they are exactly what you are looking for and the global fence misses all
66.

In [ ]:
s = orders["Sales"]
q1, q3 = s.quantile(0.25), s.quantile(0.75)
global_mask = s > q3 + 1.5 * (q3 - q1)
per_cat = orders.groupby("Category")["Sales"].transform(
    lambda x: x > x.quantile(0.75) + 1.5 * (x.quantile(0.75) - x.quantile(0.25)))

only_cat = orders[per_cat & ~global_mask]
print("rows:", len(only_cat))
print(only_cat.nlargest(8, "Sales")[["OrderID", "Category", "Sales"]]
      .to_string(index=False))
print()
print("global fence: %.2f" % (q3 + 1.5 * (q3 - q1)))

### Question 6

99th percentile `12483.55`. -> `11` orders (`1.0%` of rows) carrying **`182762.50`, `11.4%` of revenue**.

A percentile rule has one honest property the other two lack: **you choose
the count in advance.** 'The top 1%' is 11 rows by construction, whatever
the distribution does.

That makes it predictable and stable over time, which matters for anything
that runs on a schedule. The IQR and z-score thresholds move as the data
moves, so the same rule flags a different proportion every month.

In [ ]:
s = orders["Sales"]
thresh = s.quantile(0.99)
mask = s > thresh
print("99th percentile: %.2f" % thresh)
print("orders above it: %d (%.1f%% of rows)" % (mask.sum(), 100 * mask.sum() / len(s)))
print("their revenue:   %.2f (%.1f%% of the total)"
      % (s[mask].sum(), 100 * s[mask].sum() / s.sum()))

### Question 7

IQR `123` rows (`11.3%`) / `56.8%` of revenue · `|z| > 3` `27` rows (`2.5%`) / `21.8%` · top 1% `11` rows (`1.0%`) / `11.4%`.

Three standard rules on one column: **123, 27 and 11 rows**, removing 57%,
22% and 11% of revenue.

An order-of-magnitude spread in what counts as an outlier, and none of the
three is wrong. The choice is not in the data — it is a decision about how
much of the tail you are willing to call unusual, and it should be made
deliberately and written down.

Notice that all three remove disproportionately more revenue than rows, by a
factor of about five in every case. That is inevitable when the thing you are
filtering on is the thing you are measuring, and it is the reason to flag
rather than delete.

In [ ]:
s = orders["Sales"]
q1, q3 = s.quantile(0.25), s.quantile(0.75)
rules = {
    "IQR 1.5x":  s > q3 + 1.5 * (q3 - q1),
    "|z| > 3":   ((s - s.mean()) / s.std()).abs() > 3,
    "top 1%":    s > s.quantile(0.99),
}
for name, mask in rules.items():
    print("%-10s %4d rows (%4.1f%%)  %12.2f revenue (%4.1f%%)"
          % (name, mask.sum(), 100 * mask.sum() / len(s),
             s[mask].sum(), 100 * s[mask].sum() / s.sum()))

### Question 8

`quantile(0.75)` on a text column -> **raises** `ArrowNotImplementedError: Function 'quantile' has no kernel matching input types (large_string)`.

An unusual message, and it tells you something about this Pandas: the error
comes from **Arrow**, not from Pandas or NumPy, because pandas 3 backs its
string columns with Arrow arrays.

The practical consequence is that error messages on text columns now
sometimes name a layer you did not know you were using, and searching for
them turns up Arrow documentation rather than Pandas. `ArrowNotImplementedError`
almost always means 'you asked for a numeric operation on a string column'.

In [ ]:
print("Category dtype:", orders["Category"].dtype)
print(orders["Category"].quantile(0.75))